# IIF ESG company screener

This notebook uses the same report grouping, PDF extraction, full-text news screening, scoring, and export modules as `app.py`. Add annual report paths and optional standalone BRSR paths in the configuration cell. Each company must have one annual report; a BRSR attaches to its annual-report ticker and is scored as additional evidence.

The notebook is a local workflow. It does not upload documents to a hosted service. News search and FinBERT are optional because they require network access and model downloads.

In [ ]:
from pathlib import Path
from datetime import date
import uuid

import pandas as pd

from config import DEFAULTS
from esg.database import save_evidence_batch, save_score_batch
from esg.documents import group_documents
from esg.exporter import export_results
from esg.keyword_matcher import KeywordMatcher
from esg.pipeline import process_pdf_bytes
from esg.streamlit_adapter import records_to_score_evidence
import esg_engine

# Edit these inputs. BRSR entries are optional and must map to an annual-report ticker.
ANNUAL_REPORTS = [
    # {"path": Path(r"C:\reports\annual_report.pdf"), "ticker": "ABC.NS",
    #  "company_name": "Example Bank", "aliases": ["Example"], "fiscal_year": "FY2025"},
]
BRSR_REPORTS = [
    # {"path": Path(r"C:\reports\brsr.pdf"), "ticker": "ABC.NS",
    #  "company_name": "Example Bank", "fiscal_year": "FY2025"},
]
USE_OCR = False
EXTRACT_TABLES = False
RUN_NEWS = False  # Set True to query GDELT/Google News and load ProsusAI/finbert.
OUTPUT_DIR = Path("outputs")
RUN_ID = uuid.uuid4().hex

In [ ]:
def _document_rows(entries):
    rows = []
    for entry in entries:
        path = Path(entry["path"]).expanduser()
        rows.append({
            "file_name": path.name, "ticker": entry.get("ticker", ""),
            "company_name": entry.get("company_name", ""),
            "aliases": entry.get("aliases", []), "fiscal_year": entry.get("fiscal_year", ""),
            "content": path.read_bytes(),
        })
    return rows

companies = group_documents(_document_rows(ANNUAL_REPORTS), _document_rows(BRSR_REPORTS))
matcher = KeywordMatcher(esg_engine.ESG_DICTIONARY)
dataset_records, score_evidence, page_rows, extraction_errors = [], [], [], []
annual_success = set()

for company in companies:
    for document in company["documents"]:
        errors, page_status = [], []
        records = process_pdf_bytes(
            document["content"], document["source_file"], company["company_name"], matcher,
            DEFAULTS, ocr=USE_OCR, extract_tables=EXTRACT_TABLES,
            errors=errors, page_status=page_status,
        )
        if document["document_type"] == "annual_report" and page_status:
            annual_success.add(company["ticker"])
        for record in records:
            record.update(ticker=company["ticker"], document_type=document["document_type"],
                          fiscal_year=document["fiscal_year"])
        dataset_records.extend(records)
        score_evidence.extend(records_to_score_evidence(
            records, company["ticker"], company["company_name"], document["source_file"],
            document_type=document["document_type"], fiscal_year=document["fiscal_year"],
        ))
        page_rows.extend({"ticker": company["ticker"], "document_type": document["document_type"], **row}
                         for row in page_status)
        extraction_errors.extend({"ticker": company["ticker"], "document_type": document["document_type"], **row}
                                 for row in errors)

companies = [company for company in companies if company["ticker"] in annual_success]
if not companies:
    raise RuntimeError("No annual report yielded readable pages. Review extraction errors and OCR settings.")
score_companies = [{"ticker": c["ticker"], "company_name": c["company_name"],
                    "report_name": c["report_name"]} for c in companies]
evidence = pd.DataFrame(score_evidence)
document_scores = esg_engine.calculate_document_scores(evidence, score_companies)
print(f"Processed {len(companies)} companies and {len(dataset_records)} ESG dataset rows")

In [ ]:
news_frames, coverage_frames, search_errors = [], [], []
if RUN_NEWS:
    from transformers import pipeline
    import torch

    torch.set_num_threads(min(2, torch.get_num_threads()))
    classifier = pipeline("text-classification", model="ProsusAI/finbert",
                          tokenizer="ProsusAI/finbert", device=-1)
    for company in companies:  # exactly one discovery scan per grouped company
        candidates, errors, coverage = esg_engine.scan_news_for_company(company, date.today())
        if not candidates.empty:
            news_frames.append(candidates)
        if not coverage.empty:
            coverage_frames.append(coverage)
        if not errors.empty:
            search_errors.extend(errors.to_dict(orient="records"))
    candidates = pd.concat(news_frames, ignore_index=True) if news_frames else pd.DataFrame()
    news = esg_engine.classify_news(candidates, classifier) if not candidates.empty else candidates
else:
    news = pd.DataFrame()

incidents = pd.DataFrame(columns=["ticker", "pillar", "summary", "status", "source_urls",
                                  "official_record_url", "severity_points", "analyst_confirmed",
                                  "already_in_metric"])
rankings, pillar_detail = esg_engine.apply_incidents_and_rank(document_scores, incidents, news)
display(rankings)
display(pillar_detail)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
exports = export_results(dataset_records, extraction_errors, OUTPUT_DIR / RUN_ID)
save_evidence_batch(RUN_ID, news, evidence, dataset_records)
save_score_batch(RUN_ID, rankings, pillar_detail, incidents=incidents)
print("Saved dataset exports:", {name: str(path) for name, path in exports.items()})
print("Saved score run:", RUN_ID)
display(pd.DataFrame(page_rows))
display(pd.DataFrame(extraction_errors + search_errors))

## Review notes

- Each dataset row includes source file, page, document type, fiscal year, topic, matched statement, and extracted metrics. Review the cited source page before relying on a value.
- BRSR is optional. It is grouped with the same company's annual report and does not trigger an extra news search.
- News scoring requires a usable full article body, company-linked ESG conduct, English text, and successful scoring of every model chunk. Unavailable or rejected bodies do not fall back to headlines.
- Below-target news coverage receives a neutral news component. A ranking is a provisional analyst-review aid, not a verified ESG performance rating.
- This notebook uses the local SQLite database configured by `IIF_ESG_DB_PATH`, or the project's default database under `data/`.